# Getting Started with Zipminator

This notebook demonstrates the core Zipminator workflow: generating Kyber768 keypairs,
encrypting and decrypting messages, and applying DataFrame anonymization.

**Prerequisites**: Install the `zip-pqc` environment and run `maturin develop` to build
the Rust bindings.

In [ ]:
# Generate a Kyber768 keypair
from zipminator._core import keypair, encapsulate, decapsulate

pk, sk = keypair()

print(f"Public key size:  {len(pk.to_bytes())} bytes")
print(f"Secret key size:  {len(sk.to_bytes())} bytes")
print(f"Public key (hex): {pk.to_bytes()[:16].hex()}...")

In [ ]:
# Encrypt and decrypt a message using KEM + AES-256-GCM
from hashlib import sha3_256
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

# KEM encapsulation: sender gets ciphertext + shared secret
ct, shared_secret_sender = encapsulate(pk)
print(f"Ciphertext size:    {len(ct.to_bytes())} bytes")
print(f"Shared secret (tx): {shared_secret_sender.hex()[:32]}...")

# KEM decapsulation: receiver recovers the same shared secret
shared_secret_receiver = decapsulate(ct, sk)
print(f"Shared secret (rx): {shared_secret_receiver.hex()[:32]}...")
assert shared_secret_sender == shared_secret_receiver, "Shared secrets must match!"
print("Shared secrets match.")

# Derive AES-256 key from shared secret
aes_key = sha3_256(shared_secret_sender).digest()

# Encrypt a message
aesgcm = AESGCM(aes_key)
nonce = os.urandom(12)
message = b"Hello from the post-quantum world!"
ciphertext = aesgcm.encrypt(nonce, message, None)
print(f"\nPlaintext:  {message.decode()}")
print(f"Ciphertext: {ciphertext[:24].hex()}... ({len(ciphertext)} bytes)")

# Decrypt
plaintext = aesgcm.decrypt(nonce, ciphertext, None)
print(f"Decrypted:  {plaintext.decode()}")

In [ ]:
# DataFrame encryption with AdvancedAnonymizer (Level 1)
import pandas as pd
from zipminator.anonymizer import AdvancedAnonymizer

anonymizer = AdvancedAnonymizer()

df = pd.DataFrame({
    "name": ["Alice Johnson", "Bob Smith", "Carol Williams"],
    "email": ["alice@example.com", "bob@corp.net", "carol@hospital.org"],
    "phone": ["555-123-4567", "555-987-6543", "555-246-8135"],
    "salary": [85000, 92000, 78000],
})

print("Original DataFrame:")
print(df.to_string(index=False))
print()

result = anonymizer.anonymize(df, level=1)
print("Level 1 Anonymized (Minimal Masking):")
print(result.to_string(index=False))